# Reliable Scientific Paper Copilot Demo Notebook

This notebook walks through one reproducible end-to-end demo: create a tiny scientific PDF, ingest it, inspect retrieved evidence, generate grounded answers, and run the evaluation pipeline.

> For portability, the notebook forces the retriever into lexical fallback mode so it does not need to download embedding models before the demo works.


## 1. Setup

Run this notebook from the repository root after installing `requirements.txt`.


In [ ]:
from __future__ import annotations

import json
import sys
import tempfile
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.retrieval.retriever as retriever_module
retriever_module.SentenceTransformer = None
retriever_module.faiss = None

from src.parsing import parse_pdf
from src.chunking import chunk_by_sections
from src.retrieval import create_retriever
from src.answering import SimpleAnswerGenerator
from src.evaluation import run_experiment

WORKDIR = Path(tempfile.mkdtemp(prefix="paper-copilot-demo-"))
WORKDIR


## 2. Create a tiny demo PDF

The helper below writes a minimal text-based PDF so the parsing and chunking path is exercised for real.


In [ ]:
def build_text_pdf(path: Path, lines: list[str]) -> Path:
    def escape_pdf_text(text: str) -> str:
        return text.replace('\\', '\\\\').replace('(', '\\(').replace(')', '\\)')

    content_lines = ['BT', '/F1 12 Tf', '50 760 Td']
    first = True
    for raw_line in lines:
        if first:
            content_lines.append(f'({escape_pdf_text(raw_line)}) Tj')
            first = False
        else:
            content_lines.append('0 -18 Td')
            content_lines.append(f'({escape_pdf_text(raw_line)}) Tj')

    content_lines.append('ET')
    stream = '\n'.join(content_lines).encode('latin-1')
    objects = [
        b'1 0 obj << /Type /Catalog /Pages 2 0 R >> endobj\n',
        b'2 0 obj << /Type /Pages /Kids [3 0 R] /Count 1 >> endobj\n',
        b'3 0 obj << /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Resources << /Font << /F1 4 0 R >> >> /Contents 5 0 R >> endobj\n',
        b'4 0 obj << /Type /Font /Subtype /Type1 /BaseFont /Helvetica >> endobj\n',
        f'5 0 obj << /Length {len(stream)} >> stream\n'.encode('latin-1') + stream + b'\nendstream endobj\n',
    ]

    pdf = bytearray(b'%PDF-1.4\n')
    offsets = [0]
    for obj in objects:
        offsets.append(len(pdf))
        pdf.extend(obj)

    xref_offset = len(pdf)
    pdf.extend(f'xref\n0 {len(objects) + 1}\n'.encode('latin-1'))
    pdf.extend(b'0000000000 65535 f \n')
    for offset in offsets[1:]:
        pdf.extend(f'{offset:010d} 00000 n \n'.encode('latin-1'))

    pdf.extend(f'trailer << /Size {len(objects) + 1} /Root 1 0 R >>\nstartxref\n{xref_offset}\n%%EOF\n'.encode('latin-1'))
    path.write_bytes(pdf)
    return path

demo_lines = [
    'Reliable Scientific Paper Copilot Demo Paper',
    '',
    'Abstract',
    'This paper studies a lightweight transformer for clinical note classification in a hospital setting.',
    '',
    'Methods',
    'We train a 4 layer transformer with 8 attention heads on 12000 deidentified notes from two sites.',
    '',
    'Results',
    'The model achieves 91 percent accuracy and beats logistic regression by 7 points.',
    '',
    'Conclusions',
    'Lightweight transformers provide a strong baseline for note classification workloads.',
]

pdf_path = build_text_pdf(WORKDIR / 'demo-paper.pdf', demo_lines)
pdf_path


## 3. Ingest the paper

Parse the PDF, then run the section-aware chunker so we can inspect what the system would index.


In [ ]:
parsed = parse_pdf(str(pdf_path))
chunks = chunk_by_sections(parsed, max_chunk_size=220, overlap_size=40, max_tokens=64, overlap_tokens=12)

print('Parsed metadata:')
pprint(parsed['metadata'])
print('\nChunk summary:')
for chunk in chunks:
    preview = chunk['text'][:95].replace('\n', ' ')
    print(f"- chunk={chunk['chunk_id']} section={chunk['section']:<12} pages={chunk['metadata'].get('page_numbers')} text={preview}")


## 4. Retrieve evidence

Create a retriever over the ingested chunks and inspect the top evidence for a question.


In [ ]:
retriever = create_retriever(chunks, retrieval_mode='lexical')
question = 'How well did the model perform compared with the baseline?'
retrieved = retriever.retrieve(question, top_k=3)

for item in retrieved:
    print(json.dumps({
        'chunk_id': item['chunk_id'],
        'section': item['section'],
        'retrieval_score': round(item['retrieval_score'], 4),
        'text_preview': item['text'][:120],
    }, indent=2))


## 5. Generate a grounded answer

Use the built-in simple answer generator to answer a few paper questions with section citations.


In [ ]:
generator = SimpleAnswerGenerator(retriever)
questions = [
    'What architecture does the paper study?',
    'How many notes were used for training?',
    'How well did the model perform?',
]

for prompt in questions:
    result = generator.answer(prompt, top_k=3)
    print(f'Q: {prompt}')
    print(f"A: {result['answer']}")
    print(f"Sources: {result['sources']}")
    print('-' * 80)


## 6. Run the evaluation pipeline

The project already includes a synthetic evaluation set. Here we run it with the same lexical-only fallback settings used above so the notebook stays reproducible on a fresh machine.


In [ ]:
config_path = WORKDIR / 'demo-eval.yaml'
config_path.write_text(
    '\n'.join([
        'experiment:',
        '  name: notebook-demo-eval',
        '  pipeline_version: notebook-demo-v1',
        '  description: Demo notebook evaluation run using lexical fallback.',
        'retrieval:',
        '  mode: lexical',
        '  chunk_profile: chunking-v2',
        'answering:',
        '  generator: simple',
        'evaluation:',
        '  top_k: 5',
        '  use_answer_quality_judge: true',
    ]),
    encoding='utf-8',
)

experiment_run = run_experiment(config_path, persist_outputs=False)
aggregate = experiment_run['metrics']['aggregate']
summary = {
    'exact_match': round(aggregate['exact_match'], 4),
    'f1': round(aggregate['f1'], 4),
    'retrieval_hit': round(aggregate['retrieval_hit'], 4),
    'retrieval_mrr': round(aggregate['retrieval_mrr'], 4),
    'answer_quality': round(aggregate.get('answer_quality', 0.0), 4),
    'refusal_accuracy': round(aggregate.get('refusal_accuracy', 0.0), 4),
}
pprint(summary)
print('\nAnswerability slices:')
pprint(experiment_run['metrics'].get('slices', {}))


## 7. What to demo live next

For a portfolio walkthrough, this notebook is the shortest path to a clean story:

1. show the paper ingestion artifacts,
2. inspect retrieved evidence before answering,
3. answer one or two concrete questions,
4. close with evaluation metrics and refusal behavior.
